# NS-3.48 environment verification

Run after `00_environment_setup.ipynb` in a fresh Colab runtime. This checks the pinned source, instrumentation hashes, dependencies and build fingerprint, then invokes only NS-3 help paths to confirm the binary and scratch driver load. It does not install dependencies, clone source, apply patches, or run research simulations.

In [ ]:
from pathlib import Path
import json
import os
import sys

from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()
PROJECT_DIR = Path(
    os.environ.get('STAGE2_PROJECT_DIR', str(DRIVE_ROOT / 'DISSERTATION'))
).expanduser().resolve()
if not PROJECT_DIR.is_relative_to(DRIVE_ROOT):
    raise ValueError(f'Project folder must be under {DRIVE_ROOT}: {PROJECT_DIR}')
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Drive project folder is missing: {PROJECT_DIR}')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
MANIFEST_PATH = PROJECT_DIR / 'COLAB_ENVIRONMENT_MANIFEST.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError('Run 00_environment_setup.ipynb successfully before verification.')
SAVED_MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
print('Saved status:', SAVED_MANIFEST.get('status'))

In [ ]:
if not (PROJECT_DIR / 'colab/environment.py').is_file():
    raise RuntimeError('Setup sources are missing. Run the first cell of 00_environment_setup.ipynb successfully before verification.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from colab.environment import setup_environment

REPORT = setup_environment(
    mode='VERIFY',
    project_dir=PROJECT_DIR,
    ns3_dir=Path('/content/ns-3.48'),
)
if not REPORT['runtime']['is_colab']:
    raise RuntimeError('This notebook must be verified in an authenticated Google Colab runtime.')
if REPORT['ns3']['commit'] != 'd2add90b452d600cfb4859baed8e9ea633519447':
    raise RuntimeError('The verified source commit is not the pinned NS-3.48 commit.')
print('Environment status:', REPORT['status'])
print('Build status:', REPORT['ns3']['build_status'])
print('Build fingerprint:', REPORT['ns3']['build_fingerprint'])
print('Verified at:', REPORT['verified_at_utc'])